# YOLO y amplificadores operacionales: perros, gatos y peces
**Circuitos eléctricos y microcontroladores · Semana 4**

En este cuaderno se usa un modelo **YOLO real** (Ultralytics) para clasificar fotos de perros, gatos y peces, y se conecta el resultado con lo visto en clase:

- Cada neurona calcula $z = \sum_i w_i x_i + b$: es un **amplificador multientrada** (pesos $w_i = R_F/R_i$, sesgo con una resistencia a ±12 V).
- La no linealidad ReLU, $a = \max(0, z)$, es un **rectificador de precisión**.
- La decisión final y el umbral de confianza son **comparadores**.

> **Antes de empezar:** *Entorno de ejecución → Ejecutar todas*. Se necesita conexión a internet para instalar `ultralytics` y descargar el modelo (unos pocos MB) la primera vez. No hace falta GPU.

In [ ]:
# 1 · Instalar Ultralytics (una sola vez por sesión)
!pip -q install ultralytics

In [ ]:
# 2 · Cargar dos modelos YOLO
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

clasificador = YOLO('yolov8n-cls.pt')   # clasificación: 1000 clases de ImageNet
detector = YOLO('yolov8n.pt')           # detección: 80 clases de COCO (incluye perro y gato, no pez)
print('Clases del clasificador:', len(clasificador.names))
print('Clases del detector:', len(detector.names), '→ 15 =', detector.names[15], ', 16 =', detector.names[16])

## Agrupar las clases de ImageNet
El clasificador distingue **118 razas de perro**, **5 tipos de gato doméstico** y unas **16 especies de peces**. Sumamos las probabilidades de cada grupo:

$$P(\text{perro}) = \sum_{k=151}^{268} p_k \qquad P(\text{gato}) = \sum_{k=281}^{285} p_k \qquad P(\text{pez}) = \sum_{k \in \text{peces}} p_k$$

In [ ]:
# 3 · Índices de ImageNet de cada grupo
GRUPOS = {
    'perro': list(range(151, 269)),
    'gato': list(range(281, 286)),
    'pez': [0, 1, 2, 3, 4, 5, 6, 389, 390, 391, 392, 393, 394, 395, 396, 397],
}
for g, idx in GRUPOS.items():
    print(f'{g:>6s}: {len(idx):3d} clases, por ejemplo:', ', '.join(clasificador.names[i] for i in idx[:4]))

## Subir fotos
Ejecute la celda y elija una o varias imágenes (JPG o PNG) de perros, gatos o peces. También puede escribir enlaces directos a imágenes en la lista `ENLACES`.

In [ ]:
# 4 · Subir imágenes (en Colab) o usar enlaces directos
import os
ENLACES = [
    # 'https://...jpg',   # pegue aquí enlaces directos a imágenes, si lo prefiere
]
imagenes = []
try:
    from google.colab import files
    subidas = files.upload()
    imagenes += list(subidas.keys())
except Exception as e:
    print('Fuera de Colab: copie las imágenes a la carpeta de trabajo y escriba sus nombres en la lista.', e)
imagenes += ENLACES
print('Imágenes a analizar:', imagenes)

## Clasificación y banco de comparadores
Cada probabilidad de grupo se convierte en un voltaje de **0 a 10 V** ($V = 10\,\mathrm{V}\cdot P$), como si fuera la salida de un amplificador. Un comparador por grupo, con $V_{ref} = 5\,\mathrm{V}$ (confianza de 50 %), enciende su LED si el voltaje supera la referencia; el «mayor gana» decide la clase.

In [ ]:
# 5 · Clasificar cada imagen y mostrar el banco de comparadores
from PIL import Image
import io, urllib.request

VREF = 5.0   # umbral de confianza del comparador (V)
COLORES = {'perro': '#B5793E', 'gato': '#7E7E8C', 'pez': '#F2A33A'}

def abrir(fuente):
    if str(fuente).startswith('http'):
        with urllib.request.urlopen(fuente) as r:
            return Image.open(io.BytesIO(r.read())).convert('RGB')
    return Image.open(fuente).convert('RGB')

def clasificar(fuente):
    img = abrir(fuente)
    r = clasificador(img, verbose=False)[0]
    p = r.probs.data.cpu().numpy()
    P = {g: float(p[idx].sum()) for g, idx in GRUPOS.items()}
    V = {g: 10.0 * P[g] for g in P}
    gana = max(V, key=V.get)
    top5 = [(clasificador.names[int(i)], float(p[int(i)])) for i in np.argsort(p)[::-1][:5]]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={'width_ratios': [1.1, 1]})
    ax[0].imshow(img); ax[0].axis('off')
    ax[0].set_title(f'YOLO: {gana} ({P[gana] * 100:.1f} %)' if V[gana] > VREF else f'Sin decisión (máximo {gana}: {P[gana] * 100:.1f} %)')
    ax[1].bar(list(V), list(V.values()), color=[COLORES[g] for g in V])
    ax[1].axhline(VREF, color='#D2402F', ls='--', label=f'$V_{{ref}}$ = {VREF:.1f} V')
    for i, g in enumerate(V):
        ax[1].text(i, V[g] + 0.2, f'{V[g]:.2f} V\n' + ('LED ON' if V[g] > VREF else 'off'), ha='center', fontsize=10)
    ax[1].set_ylim(0, 11.5); ax[1].set_ylabel('salida del «amplificador» (V)'); ax[1].legend(frameon=False)
    ax[1].spines[['top', 'right']].set_visible(False)
    plt.tight_layout(); plt.show()
    print('Top 5 de ImageNet:', ', '.join(f'{n} {q * 100:.1f} %' for n, q in top5))
    return P

resultados = {}
for f in imagenes:
    resultados[f] = clasificar(f)

## Detección con cajas (modelo COCO)
El detector YOLO entrenado con COCO dibuja **cajas** y confianza para perros (clase 16) y gatos (clase 15). COCO **no tiene la clase pez**: por eso para peces se usa el clasificador anterior. El parámetro `conf` es el umbral del comparador.

In [ ]:
# 6 · Detección de perros y gatos con cajas
for f in imagenes:
    img = abrir(f)
    r = detector(img, classes=[15, 16], conf=0.35, verbose=False)[0]
    plt.figure(figsize=(7, 5)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis('off')
    plt.title(f'{len(r.boxes)} detecciones (perro/gato) con conf ≥ 0,35'); plt.show()
    for b in r.boxes:
        print(f'  {detector.names[int(b.cls)]}: confianza {float(b.conf):.2f}, caja {[round(v) for v in b.xyxy[0].tolist()]}')

## El clasificador analógico de la clase
Tres neuronas (perro, gato, pez) con cinco rasgos de 0 a 1 V. Los pesos son relaciones de resistencias con $R_F = 100\,\mathrm{k\Omega}$; la tabla de resistencias está en la presentación. Mueva los rasgos y compare con la decisión de YOLO para sus fotos.

In [ ]:
# 7 · Red neuronal analógica: multientrada + ReLU + comparadores
W = {'perro': ([-1, 2, -2, -0.5, 1], -0.5), 'gato': ([2, -1, -2, 1.5, 1], -1.0), 'pez': ([-1, -1, 3, 0, -2], 0.0)}
RASGOS = ['orejas', 'hocico', 'aletas', 'bigotes', 'patas']

def red_analogica(orejas=0.3, hocico=0.9, aletas=0.0, bigotes=0.2, patas=1.0):
    x = np.array([orejas, hocico, aletas, bigotes, patas])
    z = {k: float(np.dot(w, x) + b) for k, (w, b) in W.items()}            # amplificadores multientrada
    a = {k: max(0.0, v) for k, v in z.items()}                               # rectificadores de precisión (ReLU)
    e = np.exp(np.array(list(z.values())) - max(z.values())); p = e / e.sum()  # softmax (solo para la confianza)
    gana = max(z, key=z.get)                                                 # comparadores «el mayor gana»
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.bar([k + '\n(z)' for k in z], list(z.values()), color='#C9D2D6')
    ax.bar([k + '\n(ReLU)' for k in a], list(a.values()), color=[COLORES[k] for k in a])
    ax.axhline(0, color='#1F2A33', lw=1); ax.set_ylabel('voltaje (V)')
    ax.set_title(f'Gana: {gana} · confianza {p[list(z).index(gana)] * 100:.1f} %'); ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout(); plt.show()

try:
    from ipywidgets import interact, FloatSlider
    interact(red_analogica, **{r: FloatSlider(value=v, min=0, max=1, step=0.05, continuous_update=False)
                               for r, v in zip(RASGOS, [0.3, 0.9, 0.0, 0.2, 1.0])})
except Exception:
    red_analogica()

## Preguntas para discutir
1. ¿Qué umbral $V_{ref}$ usaría si prefiere «no decidir» antes que equivocarse? ¿Qué pasa con las fotos dudosas?
2. ¿Por qué el detector COCO no encuentra peces, pero el clasificador ImageNet sí?
3. En la red analógica, ¿qué peso cambiaría para que un gato con hocico largo no se confunda con un perro? Calcule la nueva resistencia ($R = R_F/|w|$).
4. YOLO tiene millones de pesos: ¿cuántas resistencias necesitaría una sola capa de 1000 neuronas con 1280 entradas? ¿Por qué se calcula en GPU?